# Student Budget & Saving Assistant

ISYS2001 - Assessmnet 2: Programing Project



Name: Meyleang Hour

Student ID: 23276347

## Step 1: Understand the problem

My project is a Student Budget and Saving Assistant for students who want to manage their money better. Students may find it difficult to record  their spending and know how much they can save each week. My app will use their income, expenses, current savings and savings goal to calculate their weekly spending, money left over and estimated time to reach their goal. An AI chatbot will help explain the results. I chose this project because I want to spend wisely and save towards my goals.

## Step 2: Identify the inputs and outputs

Inputs



*   Weekly income in AUD.
*   Weekly expenses in AUD: rent, food, transport, entertainment and other expenses.

*   Current savings in AUD.
*   Savings goal in AUD.

*   A question for the AI chatbot.

Outputs



*   A weekly spending breakdown by category.
*   Total weekly expenses in AUD.


*   Money left after weekly expenses, or the amount overspent.
*   The remaining amount needed to reach the savings goal.

*   Estimated weeks to reach the goal, assuming all money left each week is saved.
*   An AI chatbot response explaning the user's results.

















## Step 3: Work an example by hand

Example inputs:


*   Weekly income AUD 600.
*   Weekly expenses: rent AUD 200. food AUD 100, transport AUD 40, entertainment AUD 30 and other expenses AUD 30.


*   Current savings: AUD 200.
*   Savings goal: AUD 1,000.

Calculations:


1.   Total weekly expenses = 200 + 100 + 40 + 30 + 30 = AUD 400.
2.   Money left each week = 600 - 400 = AUD 200.


3.   Amount still needed = 1,000 - 200 = AUD 800.
4.   Weeks needed = 800 ÷ 200 = 4 weeks.

The students would reach the goal in 4 weeks if they saved all AUD 200 left each week. This assumes income and expenses stay the same and ignores interest.








## Step 4: Write pseudocode

START


Get weekly income, expneses by category, current savings and savings goal.


If any amount is missing, not a valid number or negative:


   show an error message and stop the calculation.

Add all expenses categories to get total weekly expenses.
Calculate money left = weekly income - total weekly expneses.
Calculate amount needed = saving goal - current savings.

If amount needed is zero or less:
   Set amount needed and weeks needed to zero.
   Show " Savings goal already reached."
Otherwise, if money left is zero or less:
    Show "You cannot reach the goal with this weekly budget."
Otherwise:
     Calculate weeks needed = amount needed / money left.
     Round weeks needed up to a whole number.
Show the spending breakdown, total expenses, money left, amount needed and saving estimate.

When the user asks a question:
    Send their question and calculated results to the AI model.
    Ask it to explain the results as a student budgeting assistant.
    Show its reply.
    If the connection fails, show a helpful error message.
    
End

## Step 5: Write the Python code

Python

weekly_income = 600
total_expenses = 200 + 100 + 40 + 30 + 30
money_left = weekly_income - total_expenses

print("Total weekly expneses: AUD", total_expenses)
print("Money left each week: AUD", money_left)


In [1]:
# Install required packages
%pip install -q gradio requests

# Import standard libraries
from decimal import Decimal, InvalidOperation
import json
import requests
import gradio as gr
from getpass import getpass

In [2]:
from getpass import getpass

# Securely prompt for your Google AI studio API key
GEMINI_API_KEY = getpass("Please enter your Gemini API Key: ").strip()
MODEL_NAME = "gemini-2.5-flash"

if not GEMINI_API_KEY:
    raise ValueError("API key cannot be empty.")

print("API Key configured successfully")

Please enter your Gemini API Key: ··········
API Key configured successfully


In [12]:
def to_cents(value, label):
    """Safely validate money inputs and convert them to whole cents using Decimal."""
    try:
        amount = Decimal(str(value))
    except InvalidOperation:
        raise ValueError(f"{label}: enter a valid number.")

    if not amount.is_finite() or amount < 0 or amount > 1_000_000_000:
        raise ValueError(f"{label}: use no more than two decimal places.")

    return int(amount * 100)

def calculate_budget(
    income, rent, food, transport,
    entertainment, other, savings, goal
):
    """Calculate weekly spending, remaining balance, and time to reach goal."""
    labels = [
        "Income", "Rent", "Food", "Transport",
        "Entertainment", "Other", "Current savings", "Savings goal"
    ]

    values = [
        income, rent, food, transport,
        entertainment, other, savings, goal
    ]

    amounts = [
        to_cents(value, label)
        for value, label in zip(values, labels)
    ]

    total = sum(amounts[1:6])
    left = amounts[0] - total
    needed = max(0, amounts[7] - amounts[6])

    if needed == 0:
        weeks = 0
        status = "Savings goal already reached."
    elif left <= 0:
        weeks = None
        status = "The goal cannot be reached with this weekly budget."
    else:
       # Round up using whole cents and whole weeks.
       weeks = (needed + left - 1) // left
       status = f"Estimated time to reach the goal: {weeks} weeks."

    return {
        "income": amounts[0] / 100,
        "expenses": dict(
            zip(labels[1:6], [a / 100 for a in amounts[1:6]])
        ),
        "total_expenses": total / 100,
        "money_left": left / 100,
        "current_savings": amounts[6] / 100,
        "savings_goal": amounts[7] / 100,
        "amount_needed": needed / 100,
        "weeks": weeks,
        "status": status
    }

def format_budget(result):
    """Format the calculation dictionary into a clear text summary."""
    lines = ["WEEKLY SPENDING BREAKDOWN"]

    for category, amount in result["expenses"].items():
      lines.append(f"{category}: AUD {amount:,.2f}")

    lines.append(f"\nWeekly income: AUD {result['income']:,.2f}")
    lines.append(f"Total weekly expenses: AUD {result['total_expenses']:,.2f}")

    if result["money_left"] < 0:
        lines.append(f"Overspent each week: AUD {-result['money_left']:,.2f}")
    else:
        lines.append(f"Money left each week: AUD {result['money_left']:,.2f}")

    lines.extend([
        f"Current savings: AUD {result['current_savings']:,.2f}",
        f"Savings goal: AUD {result['savings_goal']:,.2f}",
        f"Amount still needed: AUD {result['amount_needed']:,.2f}",
        result["status"],
        "",
        "Assumptions: all weekly money left is saved; income and "
        "expenses stay the same; interest is ignored."
    ])

    return "\n".join(lines)

In [8]:
def ask_finance_assistant(
    income, rent, food, transport, entertainment, other,
    savings, goal, user_question
):
    """Sends the budget data and user question to the Gemini API with a persona."""
    if not GEMINI_API_KEY:
        return "Error: Gemini API key is missing."

    try:
        result = calculate_budget(
            income, rent, food, transport, entertainment, other, savings, goal
        )
        budget_summary = format_budget(result)
    except ValueError as e:
        return f"Input Error: {e}"

    if not user_question or user_question.strip() == "":
        user_question = "Can you summarize my financial situation and give me one actionable tip?"

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={GEMINI_API_KEY}"
    headers = {"Content-Type": "application/json"}

    system_instruction = (
        "You are 'PocketPal', a friendly, encouraging, and financially sensible "
        "student budget assistant. Your job is to help students understand their "
        "weekly spending, encourage smart saving habits, and give realistic advice "
        "based strictly on the user's provided numbers. Keep your tone supportive, "
        "concise, and clear."
    )

    payload = {
        "systemInstruction": {
            "parts": [{"text": system_instruction}]
        },
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": f"Here is my financial breakdown:\n{budget_summary}\n\nMy Question: {user_question}"
                    }
                ]
            }
        ]
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)

        if response.status_code != 200:
            try:
                response_data = response.json()
                error_message = response_data.get("error", {}).get("message", "Unknown API error")
            except Exception:
                error_message = f"HTTP Status {response.status_code}"
            return f"API Error: {error_message}"

        response_data = response.json()
        candidate = response_data.get("candidates", [])[0]
        reply = candidate.get("content", {}).get("parts", [])[0].get("text", "No response generated.")
        return reply

    except Exception as e:
        return f"Connection failed: {e}"

In [9]:
 import gradio as gr

def gradio_wrapper(
    income, rent, food, transport, entertainment, other,
    savings, goal, user_question
):
    """Wrapper to link the custom calculation tool and AI chat to Gradio inputs/outputs."""
    try:
        result = calculate_budget(
            income, rent, food, transport, entertainment, other, savings, goal
        )
        summary_text = format_budget(result)

        ai_response = ask_finance_assistant(
            income, rent, food, transport, entertainment, other,
            savings, goal, user_question
        )

        return summary_text, ai_response
    except ValueError as e:
        return f"Error: {e}. Please fix the input values above to chat with the assistant.", ""

# Build the Gradio UI application
app = gr.Interface(
    fn=gradio_wrapper,
    inputs=[
        gr.Number(label="Weekly Income (AUD)", value=600),
        gr.Number(label="Rent (AUD)", value=200),
        gr.Number(label="Food (AUD)", value=100),
        gr.Number(label="Transport (AUD)", value=40),
        gr.Number(label="Entertainment (AUD)", value=30),
        gr.Number(label="Other Expenses (AUD)", value=30),
        gr.Number(label="Current Savings (AUD)", value=200),
        gr.Number(label="Savings Goal (AUD)", value=1000),
        gr.Textbox(label="Ask the AI Budget Assistant a question", value="How am I tracking towards my goal?")
    ],
    outputs=[
        gr.Textbox(label="Budget Breakdown & Analysis", lines=10),
        gr.Textbox(label="PocketPal AI Assistant Advice", lines=6)
    ],
    title="Student Budget & Saving Assistant",
    description="Enter your weekly income, expenses, and savings goals to calculate your financial health and get personalized AI advice."
)

# Launch the interface live in Colab
app.launch(inline=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://12157ba432f6906ec9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 6: Automated Testing & Assertions

In [13]:
print("Running automated tests...")

# Test 1: Normal calculation case
test_res = calculate_budget(600, 200, 100,40, 30, 30, 200, 1000)
assert test_res["total_expenses"] == 400.0,  f"Expected 400, got {test_res['total_expenses']}"
assert test_res["money_left"] == 200.0, f"Expected 200, got {test_res['money_left']}"
assert test_res["weeks"] == 4, f"Expected 4 weeks, got {test_res['weeks']}"
print("Test 1 Passed: Normal budget calculation is correct.")

# Test 2: Goal already reached case
goal_reached_res = calculate_budget(500, 100, 50, 20, 10, 20, 1500, 1000)
assert goal_reached_res["weeks"] == 0
assert "already reached" in goal_reached_res["status"]
print("Test 2 Passed: Goal already reached handled correct.")

# Test 3: Overspent / negative money left case
overspend_res = calculate_budget(300, 250, 100, 50, 20, 20, 0, 500)
assert overspend_res["weeks"] is None
assert "cannot be reached" in overspend_res["status"]
print("Test 3 Passed: Overspent / impossible goal handled correctly.")

# Test 4: Invalid negative input validation (Error handling test)
try:
    calculate_budget(-100, 200, 100, 40, 30, 30, 200, 1000)
    assert False, "Should have rasied ValueError for negative income"
except ValueError:
    print("Test 4 Passed: Negative input successfully caught and rejected.")
print("\nAll tests passed successfully! Your project code is complete.")


Running automated tests...
Test 1 Passed: Normal budget calculation is correct.
Test 2 Passed: Goal already reached handled correct.
Test 3 Passed: Overspent / impossible goal handled correctly.
Test 4 Passed: Negative input successfully caught and rejected.

All tests passed successfully! Your project code is complete.
